# Qwen inference optimization: prefix caching and 4-bit quantization

**You are here → 01 — Qwen: caching and quantization.**

**Main workflow:** understand a local AI request → optimize Qwen → optimize Nemotron.
**Optional extensions:** model-specific NIM deployment; separate CPU portability branch.

| **Step** | **Notebook** | **Purpose** |
|---|---|---|
| 0. Understand the request | [00 — How a local AI request works](00_local_serving.ipynb) | Follow a question through schema context, model serving and generated SQL, then evaluate the results to establish a quality baseline. |
| **→ 1. Qwen optimization (current)** | [01 — Qwen: caching and quantization](01_qwen_cache_quantization.ipynb) | Compare FP16/AWQ W4A16 and prefix caching, with cold and warm requests reported separately. |
| 2. Nemotron optimization | [02 — Nemotron optimization](02_nemotron_optimization.ipynb) | Test schema context, supported prefix caching, concurrent requests, scheduler capacity and output budgets. |
| 3. Qwen NIM (optional) | [03 — Qwen through NIM](03_qwen_nim.ipynb) | Deploy Qwen in a container, check the plan format, and compare schema context and concurrent requests. |
| 4. Nemotron NIM (optional) | [04 — Nemotron through NIM](04_nemotron_nim.ipynb) | Deploy the hybrid model through NIM and compare schema context and request concurrency. |
| Optional CPU portability | [05 — Qwen with ExecuTorch](05_edge_portability_executorch.ipynb) | Follow Qwen3-4B conversion and CPU export, then inspect a generated SQL answer and its timing. |
| Optional compatibility study | [06 — Nemotron with ExecuTorch](06_nemotron_executorch.ipynb) | Check whether the installed exporter accepts Nemotron and identify what a working port still needs. |

In chapter 00, we followed a question from prompt to checked SQL answer. Here, we test two optimizations for Qwen3-4B served through vLLM:
- Prefix caching: reuse computation for instructions and schema text shared across questions.
- 4-bit quantization: compress model weights using AWQ while keeping activations at FP16.
We run four configurations—FP16 or AWQ weights, each with caching off or on—and compare response times alongside SQL correctness.

You can work through this chapter in two ways. **Read the completed experiment:**
choose **Run All** in the [notebook environment](../SETUP.md) to display saved
measurements. **Run a new experiment:** use the [Live setup](../tutorial/RUN_ALL.md)
with a compatible GPU. The default reads results only.

The GPU measurements in this chapter used an **NVIDIA Quadro RTX 5000 with
16 GiB VRAM**. We use synthetic clinical-trial questions to connect speed with
SQL outcomes.

## 1. Choose how to run the experiment

The next cell loads the configuration. Check the printed mode so you know
whether later cells are reading a completed run or collecting new results.

| Mode | What the cells do |
|---|---|
| **Replay (default)** | Read the supplied results and rebuild their tables; no inference runs |
| Plan | Show settings and commands for a new experiment; no inference runs |
| Live | Start the configured server, make requests and save new measurements |

For Plan or Live, create `tutorial/config/vllm-qwen.local.json` using
[the run guide](../tutorial/RUN_ALL.md). Put your paths and settings there. Editing
that config affects a new run; it doesn't change the measurements read in Replay.

We use `Qwen/Qwen3-4B` through native vLLM, with weights loaded as **FP16**.
AWQ uses four-bit weights and FP16 activations; it does not quantize the KV cache.

There are **four distinct questions**, repeated five times for serving measurements.
The separate application evaluation includes planning, SQL checks, execution, and up
to one repair. Twenty repeated requests are not twenty independent quality examples.


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/clinical_trial_mvp').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from IPython.display import JSON, Markdown, display
from clinical_trial_mvp.tutorial.workflow import Study
from clinical_trial_mvp.tutorial.teaching import server_example, request_example, response_example, interpret
if 'study' in globals():
    study.close()
study = Study('vllm', 'qwen', root=ROOT)
results = study.results
print('Mode:', study.mode)
print('Model:', 'Qwen/Qwen3-4B')
print(study.prepare())


Mode: replay
Model: Qwen/Qwen3-4B
Replay: recorded Dell measurements; no model download, server, GPU or network calls.


## 2. Two optimizations, two different jobs

An LLM first processes the input (**prefill**), then generates new tokens one
at a time (**decode**). Prefix caching reuses work for identical input prefixes,
so we expect its main benefit in the time to first token, not the speed of
generating every later token.

AWQ W4A16 changes the weight representation: four-bit weights with FP16 activations.
It can reduce weight storage, but whether it improves latency depends on the
model, hardware and kernels. It does not make the activations or KV cache four-bit.

We'll test these effects separately and together. The application will still
check plans, SQL execution and reference results; an HTTP success is not an answer-quality score.


Here's our experiment matrix:

| | Prefix cache off | Prefix cache on |
|---|---|---|
| FP16 weights | Our baseline | Reuse input-prefix work |
| AWQ W4A16 | Change the weight representation | Combine both changes |

We keep the schema, four questions, seed 42, temperature 0, output cap 1024,
and context limit 8192 fixed. The live workflow requires an AWQ SQL compatibility
check before running the matrix.

To test caching fairly, each target follows this sequence:

**Reset → cold target → reset → prime with a different question sharing the same
schema prefix → repeat the identical target warm.**

Here, “cold” means an empty prefix cache on an already loaded and warmed engine,
not a cold server start. Reset must confirm `success=true`. Startup, readiness,
warmup, reset, priming and SQL scoring are outside individual request timing.
The launcher enables `VLLM_SERVER_DEV_MODE=1` for reset access on the local server;
the [cache-reset case study](../tutorial/CASE_STUDIES.md) explains why this matters.


## 3. Check that the model can complete a SQL request

Before comparing speed, check that the AWQ checkpoint and runtime can work
with the application. This **smoke test** sends a few questions through planning,
SQL generation and execution. It can catch an integration failure before we
spend time on a larger run; it doesn't replace the later quality comparison.

In Replay, the next cell shows the saved smoke result when one is available.
Plan makes no model calls. Live runs the check and stops dependent experiments
if the required check fails. Read the status and any error before continuing.


In [2]:
smoke = study.smoke()
if isinstance(smoke, dict):
    print(smoke.get('_recorded', {}).get('label', study.mode), smoke.get('status'), smoke.get('reason', ''))
else:
    print(smoke)


recorded Dell measurement completed 


## 4. Run the comparisons

We'll establish a baseline, then change a selected setting in each comparison.
Before running a cell, note what changes and what stays fixed. Afterward, compare
request timing with the generated SQL and its result check. An experiment finishing
successfully doesn't mean every answer it produced was correct.


### Establish the FP16 baseline

Let's start without prefix caching. The next cell loads or measures the reference
configuration that we'll compare against. Check its status and SQL results before
using its latency as a target to beat.


In [3]:
result = study.run('fp16_cache_off')
print(result.get('_recorded', {}).get('label', study.mode), 'fp16_cache_off:', result['status'])
display(Markdown(interpret(study, 'fp16_cache_off')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'fp16_cache_off')))


recorded Dell measurement fp16_cache_off: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/runs/20261106T221400_918883Z.json](../experiment-tracker/mvp4/runs/20261106T221400_918883Z.json)

**Prediction:** Establish executable SQL and separate application quality from request speed.

**Observation:** Application result match: 2/4; SQL executed: 3/4. Request E2E p50/p95: 6.081/7.653 s; aggregate output: 23.952 tok/s.

**Explanation:** Transport timing and application correctness measure different stages.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Use this baseline to judge useful completed tasks, not latency alone.

### Connect the server settings to the request

Start with the server: which checkpoint is loaded, what precision does it use,
and what context and scheduling limits apply? The next cell displays those settings
and a launch command so you can connect the experiment config to the serving process.
In Plan, it shows the proposed settings for your new run.

The [vLLM launcher](../src/clinical_trial_mvp/serving/launch.py) contains the implementation.


In [4]:
display(JSON(server_example(study, 'fp16_cache_off')))


<IPython.core.display.JSON object>

**Next, inspect the request.** The model receives a prompt, generation limits
and, where enabled, a JSON response schema. Look for the shared database context,
the question-specific text and the requested output format in the next cell.
This lets us see what the application asks the model to do before judging its answer.

This inspection displays the example without sending another request.
The [request builder](../src/clinical_trial_mvp/serving/client.py) defines the payload.


In [5]:
display(JSON(request_example(study, 'fp16_cache_off')))


<IPython.core.display.JSON object>

**Finally, follow the answer.** The next cell shows the generated plan and SQL,
any repair attempt, and the comparison with the reference query. Check the
question's filters, grouping and counted population as well as the score.
Matching rows on this database is useful, but it doesn't prove that two queries
express the same logic for every possible dataset.

You can inspect the [plan and SQL checks](../src/clinical_trial_mvp/agent/analyst.py)
and [reference-result scorer](../src/clinical_trial_mvp/evaluation/scorers.py).


In [6]:
display(JSON(response_example(study, 'fp16_cache_off')))


<IPython.core.display.JSON object>

### Reuse the shared prefix

Now keep FP16 weights and enable prefix caching. Compare the cold and warm phases:
we expect warm TTFT to fall if the shared schema prefix is actually reused.
Don't mistake a warm-cache result for the latency of a first-time request.


In [7]:
result = study.run('fp16_cache_on')
print(result.get('_recorded', {}).get('label', study.mode), 'fp16_cache_on:', result['status'])
display(Markdown(interpret(study, 'fp16_cache_on')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'fp16_cache_on')))


recorded Dell measurement fp16_cache_on: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/runs/20261106T224700_600112Z.json](../experiment-tracker/mvp4/runs/20261106T224700_600112Z.json)

**Prediction:** Shared-prefix reuse should reduce warm prefill latency more than total generation time.

**Observation:** Cold → warm TTFT p50: 2.019 → 0.083 s. Request E2E p50: 6.038 → 4.039 s. Direct SQL result match: 10/20 cold; 10/20 warm.

**Explanation:** The paired phase records are consistent with avoiding repeated prefill; generation still consumes time.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Consider caching for repeated schema/system prefixes after checking both phase quality counts.

### Change the weights to AWQ W4A16

With prefix caching off again, let's change only the weight representation.
Read latency alongside generated SQL and output-token counts. AWQ may change the
answer as well as its execution path; smaller weights do not guarantee faster or better answers.


In [8]:
result = study.run('awq_w4a16_cache_off')
print(result.get('_recorded', {}).get('label', study.mode), 'awq_w4a16_cache_off:', result['status'])
display(Markdown(interpret(study, 'awq_w4a16_cache_off')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'awq_w4a16_cache_off')))


recorded Dell measurement awq_w4a16_cache_off: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/runs/20261106T230608_780309Z.json](../experiment-tracker/mvp4/runs/20261106T230608_780309Z.json)

**Prediction:** Four-bit weights reduce weight storage; latency and SQL quality may change in either direction.

**Observation:** Application result match: 2/4; SQL executed: 3/4. Request E2E p50/p95: 4.310/5.123 s; aggregate output: 34.040 tok/s.

**Explanation:** Quantization changes weight representation and kernel behavior; caching addresses prefix reuse.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Benchmark compatible AWQ when weight capacity matters; no native VRAM reduction is established here.

### Combine AWQ and prefix caching

Finally, enable caching for the AWQ checkpoint. Compare this run with AWQ/cache-off
to study prefix reuse, and with FP16/cache-on to study weight representation.
Use matched cold or warm phases rather than mixing cache conditions.


In [9]:
result = study.run('awq_w4a16_cache_on')
print(result.get('_recorded', {}).get('label', study.mode), 'awq_w4a16_cache_on:', result['status'])
display(Markdown(interpret(study, 'awq_w4a16_cache_on')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'awq_w4a16_cache_on')))


recorded Dell measurement awq_w4a16_cache_on: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/runs/20261106T232408_684730Z.json](../experiment-tracker/mvp4/runs/20261106T232408_684730Z.json)

**Prediction:** Shared-prefix reuse should reduce warm prefill latency more than total generation time.

**Observation:** Cold → warm TTFT p50: 2.083 → 0.057 s. Request E2E p50: 4.271 → 2.258 s. Direct SQL result match: 15/20 cold; 15/20 warm.

**Explanation:** The paired phase records are consistent with avoiding repeated prefill; generation still consumes time.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Consider caching for repeated schema/system prefixes after checking both phase quality counts.

## 5. Read the results together

The next tables separate application quality from direct-serving timing.
**TTFT** measures time until the first generated content; **request E2E** measures
the complete generation request. **Application E2E** also includes planning,
validation, SQL execution and repair.

Read paired cold/warm phases separately from the supplemental warmed summaries.
Human semantic review is pending, and native VRAM sampling failed in the archived
runs. We can discuss latency, but those records do not demonstrate memory savings.


In [10]:
display(Markdown(study.report()))
display(Markdown(study.cache_report()))


| Configuration/status | SQL | Result match | Semantic review | Prompt tokens | TTFT p50 s | Latency p50/p95 s | tok/s | Peak memory |
|---|---|---|---|---|---|---|---|---|
| MVP4/Optimization/fp16_cache_off · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | 2344 | 2.05 | 6.08/7.65 | 23.95 | — |
| MVP4/Optimization/fp16_cache_on · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | 2344 | 0.08 | 4.06/5.70 | 34.60 | — |
| MVP4/Optimization/awq_w4a16_cache_off · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | 2344 | 2.10 | 4.31/5.12 | 34.04 | — |
| MVP4/Optimization/awq_w4a16_cache_on · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | 2344 | 0.06 | 2.28/3.09 | 62.67 | — |

| Experiment | Application E2E p50/p95 s | Request TTFT p95 s |
|---|---|---|
| fp16_cache_off · recorded Dell measurement | 12.851/15.079 | 2.052 |
| fp16_cache_on · recorded Dell measurement | 9.004/10.428 | 0.100 |
| awq_w4a16_cache_off · recorded Dell measurement | 9.684/12.191 | 2.113 |
| awq_w4a16_cache_on · recorded Dell measurement | 5.925/7.814 | 0.076 |

| Variant | Phase | SQL match | Semantic | TTFT p50/p95 s | E2E p50/p95 s | TPOT p50 s | tok/s | Peak VRAM GiB |
|---|---|---|---|---|---|---|---|---|
| fp16_cache_off · recorded Dell measurement | cache_off | 10/20 | unreviewed | 2.005/2.014 | 5.994/7.601 | 0.028 | 24.214 | — |
| fp16_cache_on · recorded Dell measurement | cold | 10/20 | unreviewed | 2.019/2.033 | 6.038/7.644 | 0.028 | 24.074 | — |
| fp16_cache_on · recorded Dell measurement | warm | 10/20 | unreviewed | 0.083/0.102 | 4.039/5.691 | 0.028 | 34.677 | — |
| awq_w4a16_cache_off · recorded Dell measurement | cache_off | 15/20 | unreviewed | 2.070/2.078 | 4.241/5.076 | 0.015 | 34.463 | — |
| awq_w4a16_cache_on · recorded Dell measurement | cold | 15/20 | unreviewed | 2.083/2.096 | 4.271/5.110 | 0.016 | 34.238 | — |
| awq_w4a16_cache_on · recorded Dell measurement | warm | 15/20 | unreviewed | 0.057/0.077 | 2.258/3.079 | 0.016 | 62.896 | — |

### Could shorter answers explain the speed difference?

Before crediting the kernels, check how much text each variant generated.
The next cell compares output tokens by phase and question using the same paired
records. Similar token counts make the latency comparison more informative, but
do not prove identical text or identical work. The displayed decode rate is a
client-side estimate, not a GPU kernel measurement.


In [11]:
from clinical_trial_mvp.tutorial.token_analysis import analyze, table
token_analysis = analyze(study.results)
display(Markdown(table(token_analysis)))


| Configuration | Phase | Requests | Mean output tokens | Median output tokens | Estimated decode tok/s p50 |
|---|---|---|---|---|---|
| FP16 | cold | 20 | 152.00 | 143.00 | 35.14 |
| FP16 | warm | 20 | 150.75 | 140.50 | 35.17 |
| AWQ W4A16 | cold | 20 | 151.50 | 143.00 | 64.24 |
| AWQ W4A16 | warm | 20 | 151.50 | 143.00 | 64.17 |

### What the recorded Qwen runs tell us

With caching enabled, FP16 median TTFT changed from **2.019 s cold to 0.083 s warm**;
request E2E changed from **6.038 to 4.039 s**. For AWQ, those values were
**2.083 → 0.057 s** and **4.271 → 2.258 s**. That's the key cache lesson:
reusing prefill work can dramatically improve the first response without removing decode time.

Direct reference-result matches stayed at **10/20** in each FP16 phase and
**15/20** in each AWQ phase. These are repeated observations of four questions,
not evidence that AWQ generally improves accuracy.

Mean FP16/AWQ output lengths were **152/151.5 tokens cold** and **150.75/151.5 warm**.
Shorter output alone is therefore not a sufficient aggregate explanation for the
timing difference. Still, the SQL changed, semantics have not been human-reviewed,
and native memory measurements are unavailable. See the
[per-case token audit](../tutorial/RESULTS.md#output-length-before-awq-attribution).

For this workload, caching is worth testing when schema prefixes repeat. AWQ is
worth considering only after checking compatibility, quality and measured latency.
These numbers describe the archive; judge a new Live run from its own tables.


### See the prefill benefit at a glance

![Quadro RTX 5000 cold and warm request timings](../tutorial/figures/01_prefix_cache.png)

This figure shows the **saved Quadro RTX 5000 measurements**. If you run Live,
read your new measurements in the tables above. Its [data](../tutorial/figures/figure_data.json)
and [rendering script](../scripts/tutorial/render_findings.py) are included.


## 6. Close the study

Save the notebook to retain the displayed outputs. Replay leaves the underlying
measurements unchanged; Live saves new uniquely named records rather than
overwriting the archive.

Native servers stop after each experiment row. The next cleanup cell is safe
to repeat. The workflow cleans up its own servers on handled errors and interruptions,
but cannot do so after a kernel or OS crash. In that case, use the recorded
process/container identifier to stop only the server this study started.


In [12]:
print(study.close())


Study servers stopped. Saved records and notebook outputs are retained.


### Where to go next

Continue to [Nemotron optimization](02_nemotron_optimization.ipynb), or compare deployment choices in
[the results guide](../tutorial/RESULTS.md#what-should-i-choose).

For setup problems, see the [worked troubleshooting examples](../tutorial/CASE_STUDIES.md).
The [original executed notebook](history/dell-2026-11-07/local%20AI%20optimization%20-%20Qwen.ipynb) retains the original run outputs.
The [SQL review packet](../tutorial/SQL_REVIEW.md) and
[GPU workflow checklist](../tutorial/GPU_VALIDATION.md) describe the remaining validation work.
